In [1]:
import os

In [2]:
%pwd

'c:\\Users\\KiranSreehari\\Desktop\\text summarizer\\Text-Summarizer-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\KiranSreehari\\Desktop\\text summarizer\\Text-Summarizer-'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\textS\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [7]:
import sys
import site

print("Python:", sys.executable)
print("\nSite packages:")
print(site.getsitepackages())

print("\nPython paths:")
for p in sys.path:
    print(p)
    

Python: c:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\textS\Scripts\python.exe

Site packages:
['c:\\Users\\KiranSreehari\\Desktop\\text summarizer\\Text-Summarizer-\\textS', 'c:\\Users\\KiranSreehari\\Desktop\\text summarizer\\Text-Summarizer-\\textS\\Lib\\site-packages']

Python paths:
C:\Users\KiranSreehari\AppData\Local\Programs\Python\Python312\python312.zip
C:\Users\KiranSreehari\AppData\Local\Programs\Python\Python312\DLLs
C:\Users\KiranSreehari\AppData\Local\Programs\Python\Python312\Lib
C:\Users\KiranSreehari\AppData\Local\Programs\Python\Python312
c:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\textS

c:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\textS\Lib\site-packages
C:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\src


In [8]:
import sys

sys.path.insert(
    0,
    r"C:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\src"
)

In [9]:
import textsummarizer

print(textsummarizer.__file__)

C:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\src\textsummarizer\__init__.py


In [10]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )

        return data_ingestion_config

In [12]:
import os
import urllib.request as request
import zipfile
from textsummarizer.logging import logger
from textsummarizer.utils.common import get_size

In [13]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """
        zip_file_path: str 
        Extracts the zip file to the specified directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [17]:
import yaml
from textsummarizer.constants import CONFIG_FILE_PATH

print("Path:", CONFIG_FILE_PATH)

with open(CONFIG_FILE_PATH, "r") as f:
    content = yaml.safe_load(f)

print("Content:", content)
print("Type:", type(content))

Path: config\config.yaml
Content: {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}}
Type: <class 'dict'>


In [19]:
config = ConfigurationManager()

[2026-10-02 10:26:08,297: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-02 10:26:08,300: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-02 10:26:08,303: INFO: common: created directory at: artifacts]


In [20]:
data_ingestion_config = config.get_data_ingestion_config()

print(data_ingestion_config)

[2026-10-02 10:26:24,127: INFO: common: created directory at: artifacts/data_ingestion]
DataIngestionConfig(root_dir=WindowsPath('artifacts/data_ingestion'), source_URL='https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip', local_data_file=WindowsPath('artifacts/data_ingestion/data.zip'), unzip_dir=WindowsPath('artifacts/data_ingestion'))


In [21]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-02 10:26:36,847: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-02 10:26:36,851: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-02 10:26:36,853: INFO: common: created directory at: artifacts]
[2026-10-02 10:26:36,855: INFO: common: created directory at: artifacts/data_ingestion]
[2026-10-02 10:26:40,962: INFO: 2855030736: artifacts\data_ingestion\data.zip downloaded with following info: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: E9E0:D710D:2820A:4E594:6ABF3985
x-github-edge-region: centralindia
Accept-Ranges: bytes
Date: Fri, 02 Oct 2026 04:56:38 GMT
Via: 1.1 varnish
X-Served-